# PEFT Fine-Tuning: Car Rental Review Summarization & Classification
Fine-tunes `google/flan-t5-base` on `amazon_us_reviews/Automotive_v1_00` using LoRA adapters.

The model is trained on two tasks simultaneously:
- **Summarization** — condense a review into its headline
- **Classification** — assign one or more categories (Vehicle Condition, Customer Service, etc.)

> **Tip:** Go to `Runtime → Change runtime type` and select **GPU** (T4) for faster training.

In [ ]:
!pip install -q "torchao>=0.16.0" transformers datasets peft accelerate huggingface_hub

In [ ]:
import time
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

## 0. Config
Mirrors `config.py`. Change these to control training behaviour.

In [ ]:
# ── Model ──────────────────────────────────────────────────────────────────
MODEL_NAME    = "google/flan-t5-xl"
ADAPTER_PATH  = "miladghofrani/car-rental-peft-adapter-xl"

# ── Dataset ────────────────────────────────────────────────────────────────
HF_DATASET = "miladghofrani/car-rental-reviews"

# ── Training ───────────────────────────────────────────────────────────────
MAX_STEPS    = None    # None = full training run
NUM_EPOCHS   = 5
LEARNING_RATE = 1e-3

# ── LoRA ───────────────────────────────────────────────────────────────────
LORA_RANK    = 16     # reduced from 32 to fit flan-t5-xl on T4 (16GB)
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

# ── Categories ─────────────────────────────────────────────────────────────
CATEGORIES = [
    "Cleanliness",
    "Vehicle Condition",
    "Pickup Experience",
    "Return Experience",
    "Hidden Fees & Billing",
    "Insurance & Upselling",
    "Staff & Communication",
    "Booking & App",
]

## 1. Detect Device

In [ ]:
def get_device():
    print('🚀 Initializing the LLM Pipeline...')
    print('--------------------------------------------------')
    print('✅ All required libraries imported successfully.')
    print(f'✅ PyTorch version: {torch.__version__}')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'✅ Processing device set to: {device.upper()}')
    print('--------------------------------------------------')
    return device

device = get_device()

## 2. Load & Build Multi-Task Dataset
Loads `miladghofrani/car-rental-reviews` from HF Hub and builds two training examples per review:
- **Summarization**: review body → human-written summary (2-3 sentences)
- **Classification**: review body → pre-labeled categories from the dataset

In [ ]:
from datasets import DatasetDict

def load_review_dataset():
    print(f"\n📥 Loading car rental reviews from HF Hub ({HF_DATASET})...")
    raw = load_dataset(HF_DATASET, split="train")
    dataset = DatasetDict({"train": raw})

    print(f"✅ Loaded {dataset['train'].num_rows:,} reviews")
    sample = dataset["train"][0]
    print("\n🔍 Sample:")
    print(f"  Review    : {sample['review_body'][:200]}...")
    print(f"  Summary   : {sample['summary']}")
    print(f"  Categories: {sample['categories']}")
    return dataset


def build_multitask_dataset(dataset):
    print(f"\n⚙️  Building multi-task examples...")
    categories_str = ", ".join(CATEGORIES)
    valid_sentiments = {"positive", "negative", "mixed"}
    inputs, outputs = [], []

    for row in dataset["train"]:
        body       = (row.get("review_body") or "").strip()
        summary    = (row.get("summary") or "").strip()
        categories = row.get("categories") or []
        sentiment  = (row.get("sentiment") or "").strip().lower()

        if not body or not summary:
            continue

        # Summarization task
        inputs.append(f"Summarize the following car rental review.\n\n{body}\n\nSummary:")
        outputs.append(summary)

        # Classification task
        inputs.append(
            f"Classify this car rental review into one or more of these categories: "
            f"{categories_str}.\n\nReview: {body}\n\nCategories:"
        )
        outputs.append(", ".join(categories) if categories else "Staff & Communication")

        # Sentiment task (only when label is available)
        if sentiment in valid_sentiments:
            inputs.append(
                f"What is the overall sentiment of this car rental review? "
                f"Answer with exactly one word: positive, negative, or mixed.\n\n"
                f"Review: {body}\n\nSentiment:"
            )
            outputs.append(sentiment)

    split = Dataset.from_dict({"input": inputs, "output": outputs}).train_test_split(test_size=0.1, seed=42)
    print(f"✅ {split['train'].num_rows:,} train / {split['test'].num_rows:,} validation examples")
    return split


raw_dataset       = load_review_dataset()
multitask_dataset = build_multitask_dataset(raw_dataset)

In [ ]:
# Verify training data before proceeding
print("Train examples:", multitask_dataset["train"].num_rows)
print("Test examples:", multitask_dataset["test"].num_rows)
print("\nSample input:")
print(multitask_dataset["train"][0]["input"])
print("\nSample output:")
print(multitask_dataset["train"][0]["output"])

assert multitask_dataset["train"].num_rows > 3000, "Dataset too small — check internet/dataset!"

import shutil, os
shutil.rmtree(os.path.expanduser("~/.cache/huggingface/datasets"), ignore_errors=True)
print("\nCache cleared.")

## 3. Load Model & Tokenizer

In [ ]:
def print_number_of_trainable_model_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(
        f"📊 Model Parameter Report:\n"
        f"--------------------------------------------------\n"
        f"  Total Parameters:     {total:,}\n"
        f"  Trainable Parameters: {trainable:,}\n"
        f"  Percentage Trainable: {100 * trainable / total:.4f}%\n"
        f"--------------------------------------------------"
    )

def load_tokenizer(model_name=MODEL_NAME):
    print("\n🔤 Loading Tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print(f"✅ Tokenizer loaded for {model_name}.")
    return tokenizer

def load_llm_model(device, model_name=MODEL_NAME):
    print("\n🧠 Loading the LLM...")
    # T4 supports float16 (not bfloat16) — using torch_dtype halves VRAM vs float32
    target_dtype = torch.float16 if device == "cuda" else torch.float32
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=target_dtype).to(device)
    print(f"✅ {model_name} loaded using {target_dtype}.")
    return model

tokenizer      = load_tokenizer()
original_model = load_llm_model(device)
print_number_of_trainable_model_parameters(original_model)

## 4. Tokenize Dataset
Converts `input` / `output` columns into `input_ids` / `labels` for seq2seq training.

In [ ]:
def tokenize_dataset(tokenizer, dataset):
    print("\n⚙️  Tokenizing dataset for seq2seq training...")
    pad_id = tokenizer.pad_token_id

    def tokenize(batch):
        model_inputs = tokenizer(
            batch["input"], truncation=True, max_length=512
        )
        labels = tokenizer(
            batch["output"], truncation=True, max_length=128
        )
        # Replace padding in labels with -100 so the loss ignores them.
        # No padding here — DataCollatorForSeq2Seq pads dynamically per batch,
        # which is faster than padding everything to max_length upfront.
        model_inputs["labels"] = [
            [(t if t != pad_id else -100) for t in label]
            for label in labels["input_ids"]
        ]
        return model_inputs

    tokenized = dataset.map(tokenize, batched=True, remove_columns=["input", "output"])
    print("✅ Tokenization complete.")
    return tokenized

tokenized_dataset = tokenize_dataset(tokenizer, multitask_dataset)

## 5. Inject LoRA Adapters

In [ ]:
def setup_peft_lora_model(original_model):
    print("\n🪄 Injecting LoRA Adapters (PEFT)...")
    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        target_modules=["q", "v"],
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type=TaskType.SEQ_2_SEQ_LM,
    )
    peft_model = get_peft_model(original_model, lora_config)
    print("✅ LoRA adapters injected.")
    print_number_of_trainable_model_parameters(peft_model)
    return peft_model

peft_model = setup_peft_lora_model(original_model)

## 6. Train & Save

> Set `MAX_STEPS = None` in the Config cell for a full training run (removes the 1-step dry-run limit).

In [ ]:
def train_and_save_peft_model(peft_model, tokenizer, tokenized_datasets):
    run_dir = f"./peft-training-run-{int(time.time())}"

    training_args = TrainingArguments(
        output_dir=run_dir,
        auto_find_batch_size=True,
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_EPOCHS,
        logging_steps=1,
        max_steps=MAX_STEPS if MAX_STEPS is not None else -1,
        gradient_checkpointing=True,   # recompute activations to save VRAM
        fp16=True,                     # T4 supports float16, not bfloat16
    )

    mode = f"max_steps={MAX_STEPS}" if MAX_STEPS is not None else f"{NUM_EPOCHS} full epoch(s)"
    print(f"\n🏋️  Starting PEFT training ({mode})...")

    collator = DataCollatorForSeq2Seq(tokenizer, model=peft_model, padding=True)

    trainer = Trainer(
        model=peft_model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        data_collator=collator,
    )

    trainer.train()

    trainer.model.save_pretrained("./checkpoint-local")
    tokenizer.save_pretrained("./checkpoint-local")
    print("✅ Adapter weights saved to: ./checkpoint-local")

# gradient checkpointing requires this when used with PEFT
peft_model.enable_input_require_grads()

train_and_save_peft_model(peft_model, tokenizer, tokenized_dataset)
print("\n✅ Training complete!")

## 7. Push to HuggingFace Hub

Pushes both the dataset (one-time) and the trained adapter to your HF account.

**One-time setup:**
1. Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) → create a token with **write** permission
2. In Colab: click the 🔑 **Secrets** icon (left sidebar) → add secret named `HF_TOKEN` → paste your token

**First time only:** Run the "Push dataset" cell to upload `car_rental_reviews.jsonl` from your machine to HF Hub so future Colab sessions can load it.

In [ ]:
from huggingface_hub import login
import os

# Colab
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
# Kaggle
except ModuleNotFoundError:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        hf_token = os.environ.get("HF_TOKEN", "")

login(token=hf_token)
print("✅ Logged in to HuggingFace Hub")

### 7a. Upload Dataset to HF Hub (run once from your local machine)

Run this cell **once from your local machine** (not Colab) to upload `car_rental_reviews.jsonl` to HF Hub.
After that, Colab always loads it from `miladghofrani/car-rental-reviews` in Section 2.

In [ ]:
# Run this once from your LOCAL MACHINE to push the dataset to HF Hub.
# After that you don't need to run it again — Section 2 loads from HF Hub automatically.
#
# From your terminal:
#   cd /Users/milad.ghofrani/P-Projects/generative-ai
#   python3 -c "
#   from datasets import load_dataset
#   ds = load_dataset('json', data_files='data_processing/car_rental_reviews.jsonl', split='train')
#   ds.push_to_hub('miladghofrani/car-rental-reviews')
#   print('Dataset pushed!')
#   "

# ── OR run this cell in Colab after uploading the file via Files panel ──────
# from google.colab import files
# uploaded = files.upload()  # select car_rental_reviews.jsonl
# from datasets import load_dataset
# ds = load_dataset("json", data_files="car_rental_reviews.jsonl", split="train")
# ds.push_to_hub("miladghofrani/car-rental-reviews")
# print(f"✅ Dataset pushed: {ds.num_rows:,} reviews")

### 7b. Push Trained Adapter to HF Hub

In [ ]:
peft_model.push_to_hub(ADAPTER_PATH)
tokenizer.push_to_hub(ADAPTER_PATH)

print(f"✅ Adapter pushed to: https://huggingface.co/{ADAPTER_PATH}")
print(f"   Load it with: ADAPTER_PATH = '{ADAPTER_PATH}'")